# Ideas planner, second list, idea 2: is a thermal mosaic of our photos big enough to simulate high altitude?
Only a footprint check (no pipeline, CPU only). A photo at height H sees 1.6 H x 1.28 H metres (320 x 256 px, fx = 200 px). Stitch the photos by their known positions: which heights could a virtual photo cut from the mosaic reach? Per dataset: photo positions (the truth x, y in map metres), their footprints (rotated rectangles are ignored: the footprint is the circle of half the diagonal), the connected clusters of overlapping footprints, and the largest height H whose footprint fits inside the covered area of the best cluster with at least 80 % of the footprint covered.
Falsification (planned, from the planner): if the mosaic cannot hold a footprint of a 600 m photo (960 x 768 m), the mosaic route is dead and only the satellite-rendered simulation remains.

In [ ]:
import os, sys, time, gc, functools
from pathlib import Path
import numpy as np
import pandas as pd
import torch

ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "src")); os.chdir(ROOT)
T7 = ROOT / "tests/Test_Iteration_7/results"

DATA = os.environ.get("GATE_DATA", "official")          # official or nudt
POOL_PHOTOS = int(os.environ.get("POOL_PHOTOS", "500"))
TAG = os.environ.get("TAG", "official150")              # which rich dump (results/rich_dump/<TAG>) names the photos
N_LIMIT = int(os.environ.get("N_LIMIT", "0"))           # >0: only the first photos (smoke test)
print(DATA, TAG, "pool", POOL_PHOTOS, "limit", N_LIMIT)

In [ ]:
import functools
if DATA == "nudt":
    from train.datasets import photo_sets
    photo_sets.load_nudt_photos = functools.partial(photo_sets.load_nudt_photos, split=os.environ.get("NUDT_SPLIT", "test"))
from train.checks.rich_dump import RichDumpPipeline
pipeline = RichDumpPipeline(DATA, "edge_zncc", semantic_layer=False, semantic_decision=False, gamma=1.0, k_peaks=25, m_fine=5, matcher="loftr_minima",
                            min_inliers=8, ransac_pixels=None, matcher_confidence=0.22, satellite="new", map_margin_m=None, loftr_coarse_threshold=0.21,
                            dino_weight=None, headings=[0.0], scale_factors=[1.0])
pipeline.setup(POOL_PHOTOS, 7)
items, truths = pipeline.test_set.items, np.array([[float(v) for v in t] for t in pipeline.test_set.truths])
heights = np.array([float(i["rel_height"]) for i in items])
print(len(items), "photos; height %.0f-%.0f m; position span x %.0f m, y %.0f m" % (heights.min(), heights.max(), np.ptp(truths[:, 0]), np.ptp(truths[:, 1])))

In [ ]:
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from scipy.spatial import cKDTree
radius = 0.5 * np.hypot(1.6, 1.28) * heights                       # half the footprint diagonal of every photo
tree = cKDTree(truths)
pairs = tree.query_pairs(r=2 * radius.max(), output_type="ndarray")
close = np.hypot(*(truths[pairs[:, 0]] - truths[pairs[:, 1]]).T) < (radius[pairs[:, 0]] + radius[pairs[:, 1]]) * 0.8
pairs = pairs[close]
graph = coo_matrix((np.ones(len(pairs)), (pairs[:, 0], pairs[:, 1])), shape=(len(items),) * 2)
count, label = connected_components(graph, directed=False)
sizes = np.bincount(label)
big = int(np.argmax(sizes))
members = np.where(label == big)[0]
print("largest cluster of overlapping footprints: %d photos of %d (clusters: %d)" % (len(members), len(items), count))
# coverage raster of the best cluster (10 m cells)
pts = truths[members]; r = radius[members]
x0, y0 = (pts - r[:, None]).min(axis=0); x1, y1 = (pts + r[:, None]).max(axis=0)
step = 10.0
gx, gy = np.arange(x0, x1, step), np.arange(y0, y1, step)
cover = np.zeros((len(gy), len(gx)), bool)
for (px, py), rr in zip(pts, r):
    xs = slice(max(0, int((px - rr - x0) // step)), int((px + rr - x0) // step) + 1); ys = slice(max(0, int((py - rr - y0) // step)), int((py + rr - y0) // step) + 1)
    cover[ys, xs] = True
print("covered area %.2f km2, bounding box %.0f x %.0f m" % (cover.sum() * step * step / 1e6, x1 - x0, y1 - y0))
import scipy.ndimage as ndi
integral = cover.astype(float).cumsum(0).cumsum(1)
def best_fraction(h):
    w, hh = int(1.6 * h / step), int(1.28 * h / step)
    if w >= cover.shape[1] or hh >= cover.shape[0]:
        return 0.0
    box = integral[hh:, w:] - integral[:-hh, w:] - integral[hh:, :-w] + integral[:-hh, :-w]
    return float(box.max() / (w * hh))
rows = [dict(virtual_height_m=h, footprint="%d x %d m" % (1.6 * h, 1.28 * h), best_covered_fraction=round(best_fraction(h), 2)) for h in (200, 300, 400, 600, 800, 1000, 1500, 3000)]
M = pd.DataFrame(rows); M.to_csv(T7 / ("idea2b_mosaic_%s.csv" % DATA), index=False); print(M.to_string(index=False))